# Parse Force : Data Architect & Semantic Expert


**Team:** Parse Force  
  

### Features:
**Google Drive auto-save** - Protects against disconnections
**Checkpoint every 500 steps** - Can resume training
**Persistent storage** - All outputs saved to Drive
**Easy team sharing** - Just share Drive folder

### What This Does:
1. Preprocesses ECF 2.0 dataset (90-10 split)
2. Trains RoBERTa-Large for emotion classification
3. Generates logits for Member 3's meta-learner
4. Saves everything to your Google Drive automatically


---

In [ ]:
# CELL 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("\n" + "="*60)
print(" Google Drive Mounted Successfully!")
print("="*60)
print("\nAll outputs will auto-save to:")
print("  /content/drive/MyDrive/NLP_Project/")
print("\n Your progress is now protected from disconnections!")
print("="*60)

## STEP 1: Configure Project Paths

Creates folder structure in your Google Drive for persistent storage.

In [ ]:
# CELL 2: Path Configuration
import os
from pathlib import Path

# Define root project folder in Google Drive
PROJECT_ROOT = "/content/drive/MyDrive/NLP_Project"
CHECKPOINT_DIR = f"{PROJECT_ROOT}/checkpoints"
DATA_DIR = f"{PROJECT_ROOT}/data"
MODELS_DIR = f"{PROJECT_ROOT}/models/roberta_semantic"
PROCESSED_DATA_DIR = f"{PROJECT_ROOT}/processed_data"

# Create all directories
for dir_path in [CHECKPOINT_DIR, DATA_DIR, MODELS_DIR, PROCESSED_DATA_DIR]:
    os.makedirs(dir_path, exist_ok=True)

print("="*60)
print("PROJECT STRUCTURE CREATED IN GOOGLE DRIVE")
print("="*60)
print(f" Project root: {PROJECT_ROOT}")
print(f" Checkpoints (auto-save): {CHECKPOINT_DIR}")
print(f" Data: {DATA_DIR}")
print(f" Models: {MODELS_DIR}")
print(f" Processed data: {PROCESSED_DATA_DIR}")
print("="*60)
print("\n TIP: Open Google Drive and check MyDrive/NLP_Project/")

## STEP 2: Check GPU Availability



In [ ]:
# Check GPU availability
import torch

print("="*60)
print("GPU CHECK")
print("="*60)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f" GPU Available: {gpu_name}")
    print(f"   GPU Memory: {gpu_memory:.2f} GB")
    print("\n Training will take ~2-3 hours")
else:
    print("  WARNING: No GPU detected!")
    print("   Training will take 24+ hours on CPU")
    print("\n To enable GPU:")
    print("   Runtime → Change runtime type → GPU")
print("="*60)

## STEP 3: Install Dependencies

In [ ]:
!pip uninstall -y transformers accelerate peft datasets
!pip install transformers==4.46.1 accelerate==1.1.1 peft==0.13.2 datasets==3.1.0 sentencepiece tokenizers

In [ ]:
# Verify installation
import transformers
import datasets

print("Package Versions:")
print(f"   transformers: {transformers.__version__}")
print(f"   datasets: {datasets.__version__}")
print(f"   torch: {torch.__version__}")
print(f"   cuda: {torch.version.cuda if torch.cuda.is_available() else 'N/A'}")

## STEP 4: Upload Dataset



In [ ]:
from google.colab import files
import os

drive_data_path = f"{DATA_DIR}/Subtask_1_train.json"

# Check if already in Drive
if os.path.exists(drive_data_path):
    print("="*60)
    print(" Dataset Found in Google Drive!")
    print("="*60)
    print(f"   Location: {drive_data_path}")
    print("   Skipping upload...")
    print("="*60)

    # Copy to local temp for faster processing
    !mkdir -p data
    !cp "{drive_data_path}" data/Subtask_1_train.json
    print("\n Copied to local temp directory")

else:
    print("="*60)
    print("Dataset not found in Drive - Upload Required")
    print("="*60)
    print("Please upload: Subtask_1_train.json")
    print("(Download from: https://zenodo.org/records/10987652)")
    print()

    # Upload via browser
    uploaded = files.upload()

    # Save to both local and Drive
    !mkdir -p data
    for filename in uploaded.keys():
        !mv {filename} data/
        !cp data/{filename} "{DATA_DIR}/"
        print(f" Uploaded: {filename}")
        print(f"Saved to Drive: {DATA_DIR}/{filename}")

# Verify file exists
if os.path.exists('data/Subtask_1_train.json'):
    file_size_mb = os.path.getsize('data/Subtask_1_train.json') / (1024*1024)
    print(f"\n Dataset Ready! (Size: {file_size_mb:.1f} MB)")
else:
    print("\n ERROR: Dataset not found")
    print("   Please upload Subtask_1_train.json and re-run this cell")

## STEP 5: Create Shared Utilities

Defines constants and configuration shared across all team members.

In [ ]:
%%writefile utils.py
"""
Shared Utilities for Parse Force TECPE Project
ALL TEAM MEMBERS must use these same constants!
"""

# Emotion label mapping (DO NOT CHANGE - team standard)
EMOTION_LABELS = {
    'anger': 0,
    'disgust': 1,
    'fear': 2,
    'joy': 3,
    'sadness': 4,
    'surprise': 5,
    'neutral': 6
}

LABEL_TO_EMOTION = {v: k for k, v in EMOTION_LABELS.items()}
EMOTION_LIST = ['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'neutral']
NUM_LABELS = 7

# class Config:
#     """Shared configuration parameters"""
#     RANDOM_SEED = 42  # CRITICAL: Used for reproducible splits
#     CONTEXT_WINDOW = 3
#     VALIDATION_SPLIT = 0.1
#     MAX_LENGTH = 256
#     BATCH_SIZE = 4
#     LEARNING_RATE = 2e-5
#     NUM_EPOCHS = 10
#     WARMUP_STEPS = 500
#     WEIGHT_DECAY = 0.01
#     GRADIENT_ACCUMULATION_STEPS = 8
#     ROBERTA_MODEL = "roberta-large"
class Config:
    """Shared configuration parameters - FIXED FOR GPU MEMORY"""
    RANDOM_SEED = 42
    CONTEXT_WINDOW = 3
    VALIDATION_SPLIT = 0.1
    MAX_LENGTH = 256                    # ← CHANGED from 512
    BATCH_SIZE = 4                      # ← CHANGED from 32
    GRADIENT_ACCUMULATION_STEPS = 8     # ← NEW LINE
    LEARNING_RATE = 2e-5
    NUM_EPOCHS = 10
    WARMUP_STEPS = 500
    WEIGHT_DECAY = 0.01
    ROBERTA_MODEL = "roberta-base"

print(" utils.py created")

## STEP 6: Data Preprocessing Pipeline

Creates train/validation split with context windowing and saves to Drive.

In [ ]:
import json
import pandas as pd
from pathlib import Path
from typing import List, Dict
import datasets
from sklearn.model_selection import train_test_split
from utils import EMOTION_LABELS, Config

class DataPreprocessor:
    """Handles data preprocessing with Drive backup"""

    def __init__(self, data_path: str, output_dir: str = "./processed_data"):
        self.data_path = Path(data_path)
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(exist_ok=True, parents=True)
        self.raw_data = None
        self.flattened_data = []

    def load_raw_data(self) -> Dict:
        """Load JSON dataset"""
        print(f"Loading data from {self.data_path}...")
        with open(self.data_path, 'r', encoding='utf-8') as f:
            self.raw_data = json.load(f)
        print(f" Loaded {len(self.raw_data)} conversations")
        return self.raw_data

    def add_context_window(self, utterances: List[Dict], window_size: int = 3) -> List[Dict]:
        """Add conversational context to each utterance"""
        for i, utt in enumerate(utterances):
            start_idx = max(0, i - window_size)
            context_utts = utterances[start_idx:i]

            context = []
            for ctx_utt in context_utts:
                speaker = ctx_utt.get('speaker', 'Unknown')
                text = ctx_utt.get('text', '')
                context.append(f"{speaker}: {text}")

            utt['context'] = " | ".join(context) if context else ""
            utt['context_length'] = len(context)
        return utterances

    def flatten_conversations(self, add_context: bool = True, context_window: int = 3) -> List[Dict]:
        """Flatten nested conversation structure"""
        if self.raw_data is None:
            self.load_raw_data()

        print("Flattening conversations...")
        flattened = []

        for conv_idx, conversation in enumerate(self.raw_data):
            conv_id = conversation.get('conversation_ID', f'conv_{conv_idx}')
            utterances = conversation.get('conversation', [])

            if add_context:
                utterances = self.add_context_window(utterances, context_window)

            for utt_idx, utterance in enumerate(utterances):
                record = {
                    'conversation_id': conv_id,
                    'utterance_id': f"{conv_id}_utt_{utt_idx}",
                    'utterance_idx': utt_idx,
                    'speaker': utterance.get('speaker', ''),
                    'text': utterance.get('text', ''),
                    'emotion': utterance.get('emotion', 'neutral').lower(),
                    'emotion_label': EMOTION_LABELS.get(
                        utterance.get('emotion', 'neutral').lower(), 6
                    ),
                }

                if add_context:
                    record['context'] = utterance.get('context', '')
                    record['context_length'] = utterance.get('context_length', 0)

                emotion_cause_pairs = utterance.get('emotion-cause_pairs', [])
                if emotion_cause_pairs:
                    record['cause_span'] = emotion_cause_pairs[0].get('cause_span', '')
                    record['has_cause'] = 1
                else:
                    record['cause_span'] = ''
                    record['has_cause'] = 0

                flattened.append(record)

        self.flattened_data = flattened
        print(f" Created {len(flattened)} flattened records")
        return flattened

    def create_train_val_split(self, test_size: float = 0.1, random_state: int = 42):
        """Create 90-10 train-validation split"""
        if not self.flattened_data:
            self.flatten_conversations()

        print(f"Creating {int((1-test_size)*100)}-{int(test_size*100)} split...")

        # Group by conversation to avoid leakage
        conv_groups = {}
        for record in self.flattened_data:
            conv_id = record['conversation_id']
            if conv_id not in conv_groups:
                conv_groups[conv_id] = []
            conv_groups[conv_id].append(record)

        # Split conversations
        conv_ids = list(conv_groups.keys())
        train_conv_ids, val_conv_ids = train_test_split(
            conv_ids, test_size=test_size, random_state=random_state
        )

        train_data = []
        val_data = []

        for conv_id in train_conv_ids:
            train_data.extend(conv_groups[conv_id])

        for conv_id in val_conv_ids:
            val_data.extend(conv_groups[conv_id])

        print(f" Train: {len(train_data)} samples from {len(train_conv_ids)} conversations")
        print(f" Validation: {len(val_data)} samples from {len(val_conv_ids)} conversations")

        return train_data, val_data

    def save_to_hf_dataset(self, train_data: List[Dict], val_data: List[Dict]):
        """Save to HuggingFace Dataset with Drive backup"""
        print("Creating HuggingFace Dataset...")

        dataset_dict = datasets.DatasetDict({
            'train': datasets.Dataset.from_pandas(pd.DataFrame(train_data)),
            'validation': datasets.Dataset.from_pandas(pd.DataFrame(val_data))
        })

        # Save to local temp
        local_path = self.output_dir / "hf_dataset"
        dataset_dict.save_to_disk(str(local_path))
        print(f" Saved locally: {local_path}")

        # IMPORTANT: Also save to Google Drive
        drive_path = Path(PROCESSED_DATA_DIR) / "hf_dataset"
        dataset_dict.save_to_disk(str(drive_path))
        print(f" Backed up to Drive: {drive_path}")
        print("   (Safe from disconnections!)")

        return dataset_dict

print(" DataPreprocessor class defined")

In [ ]:
# Run preprocessing pipeline
print("="*60)
print("RUNNING DATA PREPROCESSING")
print("="*60)

preprocessor = DataPreprocessor('data/Subtask_1_train.json', './processed_data')
preprocessor.load_raw_data()
preprocessor.flatten_conversations(add_context=True, context_window=Config.CONTEXT_WINDOW)

train_data, val_data = preprocessor.create_train_val_split(
    test_size=Config.VALIDATION_SPLIT,
    random_state=Config.RANDOM_SEED
)

dataset_dict = preprocessor.save_to_hf_dataset(train_data, val_data)

print("\n" + "="*60)
print("✅ PREPROCESSING COMPLETE!")
print("="*60)
print(f"\nDataset saved to Drive: {PROCESSED_DATA_DIR}/hf_dataset/")
print("Members 2 & 3 can access this for their training!")

## STEP 7: RoBERTa Training with Auto-Save



In [ ]:
import numpy as np
from datasets import load_from_disk
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    EvalPrediction
)
from sklearn.metrics import (
    f1_score,
    precision_recall_fscore_support,
    classification_report
)
from utils import EMOTION_LIST, NUM_LABELS, Config

class RoBERTaSemanticExpert:
    """RoBERTa-Large trainer with Drive auto-save"""

    def __init__(self, model_name="roberta-large", output_dir="./models/roberta_semantic", max_length=512):
        self.model_name = model_name
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(exist_ok=True, parents=True)
        self.max_length = max_length

        print(f"Loading tokenizer: {model_name}")
        self.tokenizer = RobertaTokenizer.from_pretrained(model_name)
        self.model = None
        self.dataset = None
        self.trainer = None

    def load_dataset(self, dataset_path: str):
        """Load preprocessed dataset"""
        print(f"Loading dataset from {dataset_path}...")
        self.dataset = load_from_disk(dataset_path)
        print(f" Loaded {len(self.dataset['train'])} training samples")
        print(f" Loaded {len(self.dataset['validation'])} validation samples")
        return self.dataset

    def tokenize_function(self, examples):
        """Tokenize text with context"""
        inputs = []
        for i in range(len(examples['text'])):
            context = examples.get('context', [''] * len(examples['text']))[i]
            text = examples['text'][i]

            if context:
                combined = f"{context} {self.tokenizer.sep_token} {text}"
            else:
                combined = text
            inputs.append(combined)

        tokenized = self.tokenizer(
            inputs,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors=None
        )
        tokenized['labels'] = examples['emotion_label']
        return tokenized

    def prepare_dataset(self):
        """Tokenize the dataset"""
        print("Tokenizing dataset...")
        self.dataset = self.dataset.map(
            self.tokenize_function,
            batched=True,
            remove_columns=['conversation_id', 'utterance_id', 'utterance_idx',
                          'speaker', 'text', 'emotion', 'context',
                          'context_length', 'cause_span', 'has_cause']
        )
        self.dataset.set_format('torch')
        print(" Dataset tokenization complete")
        return self.dataset

    def load_model(self):
        """Load RoBERTa model"""
        print(f"Loading model: {self.model_name}")
        self.model = RobertaForSequenceClassification.from_pretrained(
            self.model_name,
            num_labels=NUM_LABELS,
            problem_type="single_label_classification"
        )
        print(f" Model loaded with {NUM_LABELS} emotion labels")
        return self.model

    def compute_metrics(self, eval_pred: EvalPrediction):
        """Compute evaluation metrics"""
        logits, labels = eval_pred.predictions, eval_pred.label_ids
        predictions = np.argmax(logits, axis=-1)

        precision, recall, f1, _ = precision_recall_fscore_support(
            labels, predictions, average='weighted', zero_division=0
        )

        per_class_f1 = f1_score(labels, predictions, average=None, zero_division=0)

        metrics = {
            'f1_weighted': f1,
            'precision_weighted': precision,
            'recall_weighted': recall,
        }

        for i, emotion in enumerate(EMOTION_LIST):
            if i < len(per_class_f1):
                metrics[f'f1_{emotion}'] = per_class_f1[i]

        return metrics

    def train(self, num_epochs=10, batch_size=32, learning_rate=2e-5):
        """Train RoBERTa with auto-save to Drive"""
        print("="*60)
        print("STARTING ROBERTA TRAINING")
        print("="*60)

        # Training arguments with Drive checkpoint saving
        training_args = TrainingArguments(
            output_dir=CHECKPOINT_DIR,  # Saves to Drive automatically!
            num_train_epochs=num_epochs,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            learning_rate=learning_rate,
            warmup_steps=Config.WARMUP_STEPS,
            weight_decay=Config.WEIGHT_DECAY,
            gradient_accumulation_steps=Config.GRADIENT_ACCUMULATION_STEPS,  #
            logging_dir=f"{CHECKPOINT_DIR}/logs",
            logging_steps=100,
            eval_strategy="steps",
            eval_steps=500,
            save_strategy="steps",
            save_steps=500,  # Auto-save every 500 steps to Drive!
            save_total_limit=2,  # Keep only 2 most recent checkpoints
            load_best_model_at_end=True,
            metric_for_best_model="f1_weighted",
            greater_is_better=True,
            # fp16=torch.cuda.is_available(),
            fp16 = False,
            report_to="none",
        )

        self.trainer = Trainer(
            model=self.model,
            args=training_args,
            train_dataset=self.dataset['train'],
            eval_dataset=self.dataset['validation'],
            compute_metrics=self.compute_metrics,
        )

        print(f"\n Checkpoints will auto-save to:")
        print(f"   {CHECKPOINT_DIR}")
        print("   Every 500 steps - protected from disconnections!\n")
        print(" Starting training... (this will take 2-3 hours)\n")

        train_result = self.trainer.train()

        # Save final model to Drive
        final_model_path = Path(MODELS_DIR) / "final_model"
        self.trainer.save_model(str(final_model_path))
        self.tokenizer.save_pretrained(str(final_model_path))

        print("\n" + "="*60)
        print(" TRAINING COMPLETE")
        print("="*60)
        print(f"Final training loss: {train_result.training_loss:.4f}")
        print(f"\n Final model saved to Drive: {final_model_path}")

        return train_result

    def evaluate(self, split='validation'):
        """Evaluate the model"""
        print(f"\nEvaluating on {split} set...")
        eval_results = self.trainer.evaluate(self.dataset[split])

        print("\n" + "="*60)
        print(f"EVALUATION RESULTS ({split.upper()})")
        print("="*60)

        for key, value in eval_results.items():
            print(f"{key}: {value:.4f}")

        # Save to Drive
        results_path = Path(MODELS_DIR) / f"eval_results_{split}.json"
        with open(results_path, 'w') as f:
            json.dump(eval_results, f, indent=2)

        print(f"\n Results saved to Drive: {results_path}")
        print("   (Use these for your mid-submission report!)")

        return eval_results

    def get_predictions_with_logits(self, split='validation'):
        """Generate predictions with logits for meta-learner"""
        print(f"\nGenerating predictions with logits for {split} set...")

        predictions = self.trainer.predict(self.dataset[split])

        logits = predictions.predictions
        pred_labels = np.argmax(logits, axis=-1)
        true_labels = predictions.label_ids

        probs = torch.nn.functional.softmax(torch.tensor(logits), dim=-1).numpy()
        confidences = np.max(probs, axis=-1)

        results_df = pd.DataFrame({
            'true_label': true_labels,
            'pred_label': pred_labels,
            'true_emotion': [EMOTION_LIST[l] for l in true_labels],
            'pred_emotion': [EMOTION_LIST[l] for l in pred_labels],
            'confidence': confidences,
        })

        # Add logits for all emotions
        for i, emotion in enumerate(EMOTION_LIST):
            results_df[f'logit_{emotion}'] = logits[:, i]

        # Add probabilities for all emotions
        for i, emotion in enumerate(EMOTION_LIST):
            results_df[f'prob_{emotion}'] = probs[:, i]

        # Save to Drive
        save_path = Path(MODELS_DIR) / f"predictions_logits_{split}.csv"
        results_df.to_csv(save_path, index=False)

        print(f"\n Logits saved to Drive: {save_path}")
        print("    Share this file with Member 3 for meta-learner training!")

        accuracy = (pred_labels == true_labels).mean()
        print(f"\nAccuracy: {accuracy:.4f}")
        print(f"Average confidence: {confidences.mean():.4f}")

        return results_df

    def generate_classification_report(self, split='validation'):
        """Generate detailed classification report"""
        print(f"\nGenerating classification report for {split} set...")

        predictions = self.trainer.predict(self.dataset[split])
        pred_labels = np.argmax(predictions.predictions, axis=-1)
        true_labels = predictions.label_ids

        report = classification_report(
            true_labels,
            pred_labels,
            target_names=EMOTION_LIST,
            digits=4
        )

        print("\n" + "="*60)
        print("CLASSIFICATION REPORT")
        print("="*60)
        print(report)

        # Save to Drive
        report_path = Path(MODELS_DIR) / f"classification_report_{split}.txt"
        with open(report_path, 'w') as f:
            f.write(report)

        print(f"\n Report saved to Drive: {report_path}")

        return report

print(" RoBERTaSemanticExpert class defined")

In [ ]:
# Run full training pipeline
print("="*60)
print("ROBERTA SEMANTIC EXPERT - FULL PIPELINE")
print("="*60)

trainer = RoBERTaSemanticExpert(
    model_name=Config.ROBERTA_MODEL,
    output_dir=str(Path(MODELS_DIR)),
    max_length=Config.MAX_LENGTH
)

# Load preprocessed dataset
trainer.load_dataset('./processed_data/hf_dataset')

# Prepare dataset (tokenization)
trainer.prepare_dataset()

# Load model
trainer.load_model()

# Train (auto-saves to Drive every 500 steps!)
trainer.train(
    num_epochs=Config.NUM_EPOCHS,
    batch_size=Config.BATCH_SIZE,
    learning_rate=Config.LEARNING_RATE
)

# Evaluate
trainer.evaluate('validation')

# Generate predictions with logits (for Member 3)
trainer.get_predictions_with_logits('validation')

# Classification report
trainer.generate_classification_report('validation')

print("\n" + "="*60)
print(" PIPELINE COMPLETE!")
print("="*60)
print(f"\nAll outputs saved to Google Drive: {PROJECT_ROOT}")
print("\nCheck your Google Drive at MyDrive/NLP_Project/")

In [ ]:
 !pip install --upgrade transformers

In [ ]:
from transformers import pipeline
import torch

model_path = "/content/drive/MyDrive/NLP_Project/models/roberta_semantic/final_model"

classifier = pipeline(
    "text-classification",
    model=model_path,
    device=0 if torch.cuda.is_available() else -1
)

# Emotion mapping
emotions = ['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'neutral']

test_texts = [
    "I am so happy!",
    "This is disgusting!",
    "I'm afraid of heights",
    "I feel sad today",
    "What a wonderful surprise!",
    "I'm really angry right now",
    "Just a normal day"
]

results = classifier(test_texts)

print("="*70)
print("EMOTION PREDICTIONS")
print("="*70)

for text, result in zip(test_texts, results):
    label_num = int(result['label'].split('_')[1])
    emotion = emotions[label_num]
    confidence = result['score']

    print(f"Text: '{text}'")
    print(f"  Label: {label_num} | Emotion: {emotion:<10} | Confidence: {confidence:.4f}")
    print()

print("="*70)


In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import torch
import gc

# Clear everything first
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

print("="*60)
print("GPU MEMORY DIAGNOSTIC")
print("="*60)
print(f"GPU Available: {torch.cuda.is_available()}")
print(f"GPU Name: {torch.cuda.get_device_name(0)}")
print(f"GPU Total Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print(f"GPU Current Memory: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
print(f"GPU Reserved Memory: {torch.cuda.memory_reserved(0) / 1e9:.2f} GB")
print(f"GPU Free Memory: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)) / 1e9:.2f} GB")
print("="*60)